---
## Cell 0 — Master Configuration

In [ ]:
import os, torch
from pathlib import Path

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

# OOM FIX: was 32
BATCH_SIZE  = 16
GRAD_ACCUM  = 2
N_WORKERS   = 2

# RAM FIX: cap cache size
MAX_CACHE_N = 17_000

SFREQ       = 200
WIN_SEC     = 50
LABEL_SEC   = 10
WIN_SAMP    = SFREQ * WIN_SEC      # 10000
LABEL_SAMP  = SFREQ * LABEL_SEC   # 2000
LABEL_START = (WIN_SAMP - LABEL_SAMP) // 2
LABEL_END   = LABEL_START + LABEL_SAMP

LL = [('Fp1','F7'),('F7','T3'),('T3','T5'),('T5','O1')]
LP = [('Fp1','F3'),('F3','C3'),('C3','P3'),('P3','O1')]
RP = [('Fp2','F4'),('F4','C4'),('C4','P4'),('P4','O2')]
RL = [('Fp2','F8'),('F8','T4'),('T4','T6'),('T6','O2')]
BIPOLAR_PAIRS = LL + LP + RP + RL
N_CH    = len(BIPOLAR_PAIRS)   # 16
CLASSES = ['seizure','lpd','gpd','lrda','grda','other']
N_CLASSES = 6

# OOM FIX: halved from [64,128,256,256]
CNN_DIMS    = [32, 64, 128, 128]
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
ATTN_HEADS  = 4
DROPOUT     = 0.3

N_FOLDS      = 3
MAX_EPOCHS   = 25
PATIENCE     = 6
LR           = 1e-3
WEIGHT_DECAY = 1e-4
MIN_VOTES    = 8

ADAPTER_DIM    = 32
ADAPTER_LR     = 5e-4
ADAPTER_EPOCHS = 8
SMOKE_N        = 50

BASE      = Path('/kaggle/input/competitions/hms-harmful-brain-activity-classification')
TRAIN_CSV = BASE / 'train.csv'
TRAIN_EEG = BASE / 'train_eegs'
TEST_EEG  = BASE / 'test_eegs'
OUT_DIR   = Path('/kaggle/working')
MODEL_DIR = OUT_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)
CACHE_NPY = OUT_DIR / 'eeg_cache.npy'
CACHE_IDS = OUT_DIR / 'eeg_ids.npy'

print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {props.total_memory/1e9:.1f} GB')
print(f'Input   : ({N_CH}, {LABEL_SAMP})  — 16 ch x 10 s @ 200 Hz')
print(f'Batch   : {BATCH_SIZE} x accum {GRAD_ACCUM} = eff. {BATCH_SIZE*GRAD_ACCUM}')
print('Config ready')

---
## Cell 1 — Imports

In [ ]:
import gc, warnings, random, copy, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler  # VERSION 2 CHANGE: added WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, confusion_matrix, classification_report,
                              roc_curve, auc as sklearn_auc)  # VERSION 2 CHANGE: added roc_curve, auc
from scipy import signal as scipy_signal
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
print('Imports done')

---
## Cell 2 — Data Audit
No training here — only verification. Catches file/schema issues early.

In [ ]:
print('Loading train.csv...')
df = pd.read_csv(TRAIN_CSV)
print(f'  Rows x cols : {df.shape}')

# GroupKFold FIX: check patient_id nulls before split
null_pid = df['patient_id'].isnull().sum()
if null_pid > 0:
    print(f'  Dropping {null_pid} rows with null patient_id')
    df = df.dropna(subset=['patient_id']).reset_index(drop=True)
assert df['patient_id'].isnull().sum() == 0

VOTE_COLS = [f'{c}_vote' for c in CLASSES]
PROB_COLS = [f'{c}_prob' for c in CLASSES]
missing_votes = [c for c in VOTE_COLS if c not in df.columns]
assert not missing_votes, f'Missing vote columns: {missing_votes}'

df['total_votes'] = df[VOTE_COLS].sum(axis=1)
assert (df['total_votes'] == 0).sum() == 0
for c in CLASSES:
    df[f'{c}_prob'] = df[f'{c}_vote'] / df['total_votes']

eeg_meta = df.groupby('eeg_id').agg(
    patient_id       = ('patient_id','first'),
    expert_consensus = ('expert_consensus', lambda x: x.mode()[0]),
    **{f'{c}_vote': (f'{c}_vote','sum') for c in CLASSES}
).reset_index()
eeg_meta['total_votes'] = eeg_meta[[f'{c}_vote' for c in CLASSES]].sum(axis=1)
for c in CLASSES:
    eeg_meta[f'{c}_prob'] = eeg_meta[f'{c}_vote'] / eeg_meta['total_votes']

eeg_all  = eeg_meta.copy()
eeg_high = eeg_meta[eeg_meta['total_votes'] >= MIN_VOTES].copy()

print(f'Patients    : {eeg_all["patient_id"].nunique():,}')
print(f'Unique EEGs : {len(eeg_all):,}')
print(f'High-qual   : {len(eeg_high):,}  (>={MIN_VOTES} votes)')

for cls, cnt in eeg_all['expert_consensus'].value_counts().items():
    print(f'  {cls:8s}: {cnt:5,} ({100*cnt/len(eeg_all):4.1f}%)')

# Parquet spot-check
sample_eid = int(eeg_all['eeg_id'].iloc[0])
sample_path = TRAIN_EEG / f'{sample_eid}.parquet'
assert sample_path.exists(), f'EEG file missing: {sample_path}'
raw = pd.read_parquet(sample_path)

print("WIN_SAMP =", WIN_SAMP)
print("Raw shape =", raw.shape)
print(raw.head())

# Bipolar channel check
missing_ch = []
for ch1, ch2 in BIPOLAR_PAIRS:
    if ch1 not in raw.columns: missing_ch.append(ch1)
    if ch2 not in raw.columns: missing_ch.append(ch2)
missing_ch = list(set(missing_ch))
if missing_ch:
    print(f'Missing channels (will zero-fill): {missing_ch}')
else:
    print('All 32 electrodes present')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('HMS Dataset Audit', fontsize=13, fontweight='bold')
cls_colors = ['#E74C3C','#E67E22','#F39C12','#27AE60','#2980B9','#8E44AD']
cnts = [eeg_all[eeg_all['expert_consensus']==c].shape[0] for c in CLASSES]
axes[0].bar([c.upper() for c in CLASSES], cnts, color=cls_colors, alpha=0.85)
axes[0].set_title('Class Distribution'); axes[0].set_ylabel('EEGs')
axes[0].tick_params(axis='x', rotation=20)

axes[1].hist(eeg_all['total_votes'].clip(upper=30), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(MIN_VOTES, color='red', ls='--', lw=2, label=f'Threshold={MIN_VOTES}')
axes[1].set_title('Expert Vote Distribution'); axes[1].legend()

pts_per_patient = eeg_all.groupby('patient_id').size()
axes[2].hist(pts_per_patient.clip(upper=30), bins=20, color='#8E44AD', edgecolor='white', alpha=0.85)
axes[2].set_title('EEGs per Patient')

plt.tight_layout()
plt.savefig(OUT_DIR/'audit.png', dpi=120, bbox_inches='tight')
plt.show()
print('Data audit complete')

---
## Cell 3 — EEG Preprocessor + Shape Assertions

In [ ]:
class EEGPreprocessor:
    def __init__(self):
        self.sos = scipy_signal.butter(
            5, [0.5, 40.0], btype='bandpass', fs=SFREQ, output='sos'
        )

    def load_and_process(self, eeg_id: int, eeg_dir: Path = TRAIN_EEG) -> Optional[np.ndarray]:
        path = eeg_dir / f'{eeg_id}.parquet'
        if not path.exists():
            return None
        try:
            raw = pd.read_parquet(path)
            return self._process(raw)
        except Exception:
            return None

    def _process(self, raw: pd.DataFrame) -> np.ndarray:
        n_raw = len(raw)

        # 1 — Bipolar channels
        data = np.zeros((N_CH, n_raw), dtype=np.float32)
        for i, (ch1, ch2) in enumerate(BIPOLAR_PAIRS):
            c1 = raw[ch1].values if ch1 in raw.columns else np.zeros(n_raw)
            c2 = raw[ch2].values if ch2 in raw.columns else np.zeros(n_raw)
            sig = (c1 - c2).astype(np.float32)
            sig = np.nan_to_num(
                sig,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )
            
            data[i] = sig

        # 2 — Extract middle 10 s (SPEED FIX: was using all 10000 samples)
        # 2 — Extract middle 10 s dynamically
        n_samples = data.shape[1]
        
        label_start = (n_samples - LABEL_SAMP) // 2
        label_end   = label_start + LABEL_SAMP
        
        data = data[:, label_start:label_end]
        data = np.nan_to_num(
        data,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
        )
        # 3 — Clip saturation
        data = np.clip(data, -1024.0, 1024.0)

        # 4 — Bandpass filter
        for i in range(N_CH):
            data[i] = scipy_signal.sosfilt(self.sos, data[i]).astype(np.float32)

        # 5 — Robust z-score
        med = np.median(data, axis=1, keepdims=True)
        mad = np.median(np.abs(data - med), axis=1, keepdims=True)
        mad = np.where(mad < 1e-6, 1.0, mad)
        data = (data - med) / (1.4826 * mad)

        # 6 — Final clip
        data = np.clip(data, -6.0, 6.0).astype(np.float32)

        # SHAPE FIX: assert every time
        assert data.shape == (N_CH, LABEL_SAMP), \
            f'Shape error: expected ({N_CH},{LABEL_SAMP}), got {data.shape}'
        data = np.nan_to_num(
            data,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
        
        assert np.isfinite(data).all()
        return data


prep = EEGPreprocessor()
eid0 = int(eeg_all['eeg_id'].iloc[0])
s0   = prep.load_and_process(eid0)

assert s0 is not None
assert s0.shape == (N_CH, LABEL_SAMP)
assert np.isfinite(s0).all()

print(f'Preprocessor verified')
print(f'  Shape : {s0.shape}')
print(f'  Range : [{s0.min():.2f}, {s0.max():.2f}]')

t = np.linspace(0, LABEL_SEC, LABEL_SAMP)
rnames = ['LL (Left Lat.)','LP (Left Para.)','RP (Right Para.)','RL (Right Lat.)']
rcols  = ['#E74C3C','#27AE60','#2980B9','#8E44AD']
fig, axes = plt.subplots(4, 1, figsize=(16, 8), sharex=True)
fig.suptitle(f'Preprocessed Bipolar EEG ID {eid0}', fontsize=12, fontweight='bold')
for ri in range(4):
    for ci in range(4):
        axes[ri].plot(t, s0[ri*4+ci] + ci*3.5, lw=0.7, alpha=0.9, color=rcols[ri])
    axes[ri].set_ylabel(rnames[ri], fontsize=8)
    axes[ri].grid(alpha=0.2); axes[ri].set_yticks([])
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(OUT_DIR/'sample_eeg.png', dpi=120, bbox_inches='tight')
plt.show()
print('Cell 3 complete')

In [ ]:
s0 = prep.load_and_process(eid0)

print("NaNs :", np.isnan(s0).sum())
print("Infs :", np.isinf(s0).sum())
print("Finite:", np.isfinite(s0).all())

---
## Cell 4 — Smart Cache + PyTorch Dataset
**V2 Changes:**
- `HMSDataset.__getitem__` uses `str(row['expert_consensus']).lower()` safely
- Seizure and LRDA samples get minority-aware augmentation (time-shift p=0.4, noise p=0.4, channel-drop p=0.2)
- General augmentation unchanged


In [ ]:
CACHE_NPY.unlink(missing_ok=True)
CACHE_IDS.unlink(missing_ok=True)

print("Old cache deleted")

In [ ]:
def build_eeg_cache(meta_df: pd.DataFrame, force: bool = False,
                    cap: int = MAX_CACHE_N) -> Tuple[np.ndarray, np.ndarray]:
    if CACHE_NPY.exists() and CACHE_IDS.exists() and not force:
        print('Loading existing cache from disk...')
        data = np.load(CACHE_NPY)
        ids  = np.load(CACHE_IDS)
        print(f'  Shape : {data.shape}  | {data.nbytes/1e9:.2f} GB')
        return data, ids

    rows = meta_df.head(cap)
    n    = len(rows)
    print(f'Building cache - {n:,} EEGs (cap={cap})...')
    data = np.zeros((n, N_CH, LABEL_SAMP), dtype=np.float32)
    ids  = rows['eeg_id'].values.copy()
    ok   = np.zeros(n, dtype=bool)

    for i, eid in enumerate(tqdm(ids, desc='Preprocessing', leave=True)):
        arr = prep.load_and_process(int(eid))
        if arr is not None:
            data[i] = arr
            ok[i]   = True

    data = data[ok]; ids = ids[ok]
    np.save(CACHE_NPY, data)
    np.save(CACHE_IDS, ids)
    print(f'Cache: {ok.sum():,} EEGs | {data.nbytes/1e9:.2f} GB')
    return data, ids


eeg_data, eeg_ids = build_eeg_cache(eeg_all, cap=MAX_CACHE_N)
id2idx = {int(eid): i for i, eid in enumerate(eeg_ids)}
print(f'Lookup table: {len(id2idx):,} EEGs indexed')


class HMSDataset(Dataset):
    PROB_COLS = [f'{c}_prob' for c in CLASSES]

    def __init__(self, meta: pd.DataFrame, cache: np.ndarray,
                 idx_map: dict, augment: bool = False, eeg_dir: Path = TRAIN_EEG):
        valid = meta['eeg_id'].apply(
            lambda x: int(x) in idx_map or (eeg_dir/f'{x}.parquet').exists()
        )
        self.meta    = meta[valid].reset_index(drop=True)
        self.cache   = cache
        self.idx_map = idx_map
        self.augment = augment
        self.eeg_dir = eeg_dir

    def __len__(self): return len(self.meta)

    # VERSION 2 CHANGE: minority-aware augmentation
    def __getitem__(self, i):
        row = self.meta.iloc[i]
        eid = int(row['eeg_id'])

        if eid in self.idx_map:
            x = self.cache[self.idx_map[eid]].copy()
        else:
            x = prep.load_and_process(eid, self.eeg_dir)
            if x is None:
                x = np.zeros((N_CH, LABEL_SAMP), dtype=np.float32)

        assert x.shape == (N_CH, LABEL_SAMP), f'Bad shape: {x.shape}'

        y = row[self.PROB_COLS].values.astype(np.float32)
        y = np.clip(y, 1e-7, 1.0)
        y /= y.sum()

        # VERSION 2 CHANGE: safe class name extraction
        cls = str(row['expert_consensus']).lower()

        if self.augment:
            x = self._aug(x, cls)
        return torch.from_numpy(x.copy()), torch.from_numpy(y), str(row['patient_id'])

    def _aug(self, x: np.ndarray, cls: str = 'other') -> np.ndarray:
        # General augmentation (unchanged from V1)
        if np.random.rand() < 0.5: x = -x
        if np.random.rand() < 0.5: x += np.random.randn(*x.shape).astype(np.float32)*0.04
        if np.random.rand() < 0.3: x[np.random.randint(N_CH)] = 0.0
        if np.random.rand() < 0.2:
            shift = np.random.randint(-100, 100)
            x = np.roll(x, shift, axis=1)

        # VERSION 2 CHANGE: minority-aware augmentation for seizure and lrda only
        # Mild values (max weight 2.0, conservative probabilities)
        if cls in ('seizure', 'lrda'):
            # Time shift: p=0.4, range ±150 samples (±0.75 s at 200 Hz)
            if np.random.rand() < 0.4:
                shift = np.random.randint(-150, 150)
                x = np.roll(x, shift, axis=1)
            # Small noise: p=0.4, std=0.02 (gentle relative to signal)
            if np.random.rand() < 0.4:
                x = x + np.random.randn(*x.shape).astype(np.float32) * 0.02
            # Channel dropout: p=0.2, zero one random channel
            if np.random.rand() < 0.2:
                x[np.random.randint(N_CH)] = 0.0
        return x


ds0 = HMSDataset(eeg_all, eeg_data, id2idx, augment=False)
x0, y0, p0 = ds0[0]
assert x0.shape == torch.Size([N_CH, LABEL_SAMP])
assert abs(y0.sum().item() - 1.0) < 1e-4
assert not torch.isnan(x0).any()
n_ds0 = len(ds0); print(f'Dataset ready : {n_ds0} samples')
del ds0, x0, y0

In [ ]:
print("NaNs :", np.isnan(eeg_data).sum())
print("Infs :", np.isinf(eeg_data).sum())
print("Min  :", np.nanmin(eeg_data))
print("Max  :", np.nanmax(eeg_data))

In [ ]:
import gc

gc.collect()

In [ ]:
torch.cuda.empty_cache()

---
## Cell 5 — Lightweight CNN-BiLSTM-Attention Model
~2M parameters, T4-safe at batch 16. **Unchanged from V1.**

In [ ]:
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=5, stride=2, drop=0.2):
        super().__init__()
        pad = kernel // 2
        self.main = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch), nn.GELU(), nn.Dropout(drop)
        )
        self.skip = (
            nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            ) if (in_ch != out_ch or stride != 1) else nn.Identity()
        )

    def forward(self, x):
        out  = self.main(x)
        skip = self.skip(x)
        if out.shape[-1] != skip.shape[-1]:
            skip = F.adaptive_avg_pool1d(skip, out.shape[-1])
        return F.gelu(out + skip)


class TemporalAttention(nn.Module):
    """Multi-head self-attention.
    ATTENTION DIM FIX: weights are (B, heads, T, T).
    Always average over head axis before further use.
    """
    def __init__(self, d: int, heads: int = 4, drop: float = 0.1):
        super().__init__()
        assert d % heads == 0
        self.h  = heads
        self.hd = d // heads
        self.sc = self.hd ** -0.5
        self.qkv = nn.Linear(d, 3*d)
        self.out = nn.Linear(d, d)
        self.drp = nn.Dropout(drop)
        self.nrm = nn.LayerNorm(d)

    def forward(self, x):
        B, T, D = x.shape
        res = x
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.hd)
        q, k, v = qkv.unbind(2)
        q = q.permute(0,2,1,3)
        k = k.permute(0,2,1,3)
        v = v.permute(0,2,1,3)
        w = F.softmax((q @ k.transpose(-2,-1)) * self.sc, dim=-1)
        w = self.drp(w)
        o = (w @ v).permute(0,2,1,3).reshape(B, T, D)
        return self.nrm(self.out(o) + res), w  # w: (B, heads, T, T)


class HMSModel(nn.Module):
    def __init__(self, n_ch=N_CH, n_cls=N_CLASSES, cnn_dims=None,
                 lstm_h=LSTM_HIDDEN, lstm_l=LSTM_LAYERS, heads=ATTN_HEADS, drop=DROPOUT):
        super().__init__()
        if cnn_dims is None: cnn_dims = CNN_DIMS

        kernels = [11, 7, 5, 3]
        layers  = []
        in_ch   = n_ch
        for out_ch, k in zip(cnn_dims, kernels):
            layers.append(ResBlock1D(in_ch, out_ch, kernel=k, stride=2, drop=drop))
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers)
        cnn_out  = cnn_dims[-1]

        self.lstm = nn.LSTM(cnn_out, lstm_h, lstm_l, batch_first=True,
                            bidirectional=True, dropout=drop if lstm_l > 1 else 0.0)
        lstm_out = lstm_h * 2

        self.attn = TemporalAttention(lstm_out, heads, drop=0.1)

        self.head = nn.Sequential(
            nn.LayerNorm(lstm_out), nn.Dropout(drop),
            nn.Linear(lstm_out, 128), nn.GELU(),
            nn.Dropout(drop/2), nn.Linear(128, n_cls)
        )
        self._init_weights()

    def forward(self, x, return_attn: bool = False):
        x = self.cnn(x)           # (B, 128, 125)
        x = x.permute(0, 2, 1)   # (B, 125, 128)
        x, _ = self.lstm(x)       # (B, 125, 256)
        x, w  = self.attn(x)      # w: (B, heads, T, T)
        x = x.mean(dim=1)         # (B, 256)
        logits = self.head(x)
        if return_attn: return logits, w
        return logits

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


torch.cuda.empty_cache()
_m  = HMSModel().to(DEVICE)
_xt = torch.randn(BATCH_SIZE, N_CH, LABEL_SAMP).to(DEVICE)

with torch.no_grad():
    _lo, _at = _m(_xt, return_attn=True)

assert _lo.shape == (BATCH_SIZE, N_CLASSES)
assert _at.shape == (BATCH_SIZE, ATTN_HEADS, 125, 125), f'Attn shape: {_at.shape}'

print(f'HMSModel verified')
print(f'  Input   : {list(_xt.shape)}')
print(f'  Logits  : {list(_lo.shape)}')
print(f'  AttnMap : {list(_at.shape)}  (B, heads, T, T)')
print(f'  Params  : {_m.n_params():,}')
del _m, _xt, _lo, _at; gc.collect(); torch.cuda.empty_cache()

---
## Cell 6 — KL Divergence Loss + Grad-Accum Trainer
**V2 Changes:**
- Original `KLDivSoftLoss` kept exactly as-is (no weighting)
- New `make_weighted_sampler()` function added for mild minority oversampling
- Sample weights: seizure=1.5, lrda=1.8, lpd=1.0, gpd=1.0, grda=1.0, other=0.8


In [ ]:
class KLDivSoftLoss(nn.Module):
    def __init__(self, T: float = 1.0):
        super().__init__()
        self.T  = T
        self.kl = nn.KLDivLoss(reduction='batchmean')

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_p = F.log_softmax(logits / self.T, dim=-1)
        tgt   = targets / (targets.sum(-1, keepdim=True) + 1e-9)
        loss  = self.kl(log_p, tgt)
        assert torch.isfinite(loss), f'KL loss is not finite: {loss.item()}'
        return loss


class EarlyStopping:
    def __init__(self, patience: int = 6, mode: str = 'min'):
        self.patience = patience; self.mode = mode
        self.best  = float('inf') if mode == 'min' else float('-inf')
        self.count = 0; self.stop = False

    def __call__(self, val: float) -> bool:
        better = (val < self.best) if self.mode == 'min' else (val > self.best)
        if better: self.best = val; self.count = 0; return True
        self.count += 1
        if self.count >= self.patience: self.stop = True
        return False


def train_one_epoch(model, loader, opt, crit, scaler, sched=None, accum: int = GRAD_ACCUM):
    model.train()
    opt.zero_grad(set_to_none=True)
    tot_loss, preds, truths = 0.0, [], []

    for step, (x, y, _) in enumerate(loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            lo   = model(x)
            loss = crit(lo, y) / accum

        scaler.scale(loss).backward()

        if (step + 1) % accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            if sched: sched.step()

        tot_loss += loss.item() * accum
        with torch.no_grad():
            preds.append(F.softmax(lo, -1).argmax(-1).cpu())
            truths.append(y.argmax(-1).cpu())

    acc = (torch.cat(preds) == torch.cat(truths)).float().mean().item()
    return {'loss': tot_loss / len(loader), 'acc': acc}


@torch.no_grad()
def eval_one_epoch(model, loader, crit):
    model.eval()
    tot_loss, all_p, all_t = 0.0, [], []
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast(enabled=USE_AMP):
            lo   = model(x)
            loss = crit(lo, y)
        tot_loss += loss.item()
        all_p.append(F.softmax(lo, -1).cpu())
        all_t.append(y.cpu())

    P  = torch.cat(all_p).numpy()
    T  = torch.cat(all_t).numpy()
    pc = P.argmax(1); tc = T.argmax(1)
    acc = (pc == tc).mean()
    aucs = []
    for c in range(N_CLASSES):
        m = (tc == c).astype(int)
        if 0 < m.sum() < len(m):
            aucs.append(roc_auc_score(m, P[:, c]))
    return {
        'kl': tot_loss / len(loader), 'acc': acc,
        'auc': float(np.mean(aucs)) if aucs else 0.0,
        'probs': P, 'pred_cls': pc, 'true_cls': tc
    }


_crit = KLDivSoftLoss()
_l1   = _crit(torch.randn(8, 6), F.softmax(torch.randn(8, 6), -1))
_l2   = _crit(torch.randn(8, 6), torch.ones(8, 6)/6)
assert torch.isfinite(_l1) and torch.isfinite(_l2)
print(f'KL loss verified  (random={_l1.item():.4f}  uniform={_l2.item():.4f})')
print(f'Grad accum = {GRAD_ACCUM}  ->  effective batch = {BATCH_SIZE*GRAD_ACCUM}')
# ─────────────────────────────────────────────────────────────────
# VERSION 2 CHANGE: Mild WeightedRandomSampler (no loss weighting)
# ─────────────────────────────────────────────────────────────────
# Class weights: slightly oversample minority classes
# Values kept ≤ 2.0 to avoid aggressive oversampling
CLASS_SAMPLE_WEIGHTS = {
    'seizure': 1.5,
    'lrda':    1.8,
    'lpd':     1.0,
    'gpd':     1.0,
    'grda':    1.0,
    'other':   0.8,
}

def make_weighted_sampler(meta_df: pd.DataFrame) -> WeightedRandomSampler:
    """
    VERSION 2 CHANGE: Build a WeightedRandomSampler from expert_consensus labels.
    Minority classes (seizure, lrda) get mild upsampling.
    Applied only during training; validation uses standard sequential loading.
    """
    cls_col = meta_df['expert_consensus'].astype(str).str.lower()
    weights = cls_col.map(CLASS_SAMPLE_WEIGHTS).fillna(1.0).values.astype(np.float64)
    sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True
    )
    return sampler


print('WeightedRandomSampler helper defined')
print('Class sample weights:', CLASS_SAMPLE_WEIGHTS)


---
## Cell 7 — Smoke Test Gate
Must pass before full training.

In [ ]:
print(eeg_all['expert_consensus'].unique())
print(CLASSES)

In [ ]:
eeg_all['expert_consensus'] = (
    eeg_all['expert_consensus']
    .astype(str)
    .str.lower()
)

eeg_high['expert_consensus'] = (
    eeg_high['expert_consensus']
    .astype(str)
    .str.lower()
)

In [ ]:
print("EEG cache stats")
print(np.isnan(eeg_data).sum())
print(np.isinf(eeg_data).sum())
print(eeg_data.min(), eeg_data.max())

In [ ]:
def run_smoke_test() -> bool:
    print('\n' + '='*58)
    print(' SMOKE TEST - Verifying pipeline before full training')
    print('='*58)

    pieces = []
    for cls in CLASSES:
        rows = eeg_all[eeg_all['expert_consensus'] == cls]
        rows = rows[rows['eeg_id'].apply(lambda x: int(x) in id2idx)]
        n = min(SMOKE_N, len(rows))
        if n > 0:
            pieces.append(rows.sample(n, random_state=42))
    subset = pd.concat(pieces).reset_index(drop=True)
    print(f'Subset : {len(subset)} samples')

    ds = HMSDataset(subset, eeg_data, id2idx, augment=False)
    ld = DataLoader(ds, batch_size=min(BATCH_SIZE, 8), shuffle=True, num_workers=0)

    x0, y0, _ = next(iter(ld))
    assert x0.shape[1:] == torch.Size([N_CH, LABEL_SAMP]), f'Shape: {x0.shape}'
    print(f'  Shape   : {list(x0.shape)}  OK')
    assert not torch.isnan(x0).any() and not torch.isinf(x0).any()
    print(f'  NaN/Inf : none  OK')
    label_sums = y0.sum(dim=1)
    assert (label_sums - 1.0).abs().max() < 1e-4
    print(f'  Label sum : {label_sums.mean():.6f}  OK')

    print('Mini-training 3 epochs...')
    set_seed(0)
    model  = HMSModel().to(DEVICE)
    opt    = AdamW(model.parameters(), lr=LR)
    crit   = KLDivSoftLoss()
    scaler = GradScaler(enabled=USE_AMP)
    losses = []

    for ep in range(3):
        r = train_one_epoch(model, ld, opt, crit, scaler, accum=1)
        losses.append(r['loss'])
        print(f'  ep{ep+1}: loss={r["loss"]:.4f}  acc={r["acc"]:.3f}')

    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated()/1e9
        total = torch.cuda.get_device_properties(0).total_memory/1e9
        pct   = 100*alloc/total
        print(f'VRAM used : {alloc:.2f} / {total:.1f} GB  ({pct:.0f}%)')

    del model; gc.collect(); torch.cuda.empty_cache()
    print('SMOKE TEST PASSED')
    return True

smoke_ok = run_smoke_test()

---
## Cell 8 — GroupKFold Training (Quality-Aware, Patient-Safe)
**V2 Changes:**
- `WeightedRandomSampler` used for Stage A and B training loaders (training only)
- Per-fold: confusion matrix saved as `fold{N}_confusion.png`
- Per-fold: prediction CSV saved as `fold{N}_predictions.csv`
- Per-fold: training curves saved as `training_curves_fold{N}.png`
- All other logic (GroupKFold, Stage A/B, EarlyStopping) unchanged


In [ ]:
assert smoke_ok, 'Smoke test did not pass!'

# ─────────────────────────────────────────────────────────────────────
# VERSION 2 CHANGE: per-fold diagnostic helpers
# ─────────────────────────────────────────────────────────────────────

def save_fold_confusion(true_cls, pred_cls, fold_n: int):
    """VERSION 2 CHANGE: save confusion matrix for this fold."""
    cm = confusion_matrix(true_cls, pred_cls)
    cm_norm = cm.astype(float) / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    cls_U = [c.upper() for c in CLASSES]
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(cls_U, rotation=15)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(cls_U)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Fold {fold_n} Confusion Matrix (normalised)')
    plt.colorbar(im, ax=ax, fraction=0.04)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            v = cm_norm[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if v > 0.5 else 'black')
    plt.tight_layout()
    path = OUT_DIR / f'fold{fold_n}_confusion.png'
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.close()
    print(f'  Confusion matrix saved: {path.name}')


def save_fold_predictions(va_df: pd.DataFrame, probs: np.ndarray,
                          pred_cls: np.ndarray, true_cls: np.ndarray, fold_n: int):
    """VERSION 2 CHANGE: save per-sample predictions for error analysis."""
    out = va_df[['eeg_id', 'patient_id', 'expert_consensus']].copy().reset_index(drop=True)
    out['true_class']    = [CLASSES[t] for t in true_cls]
    out['predicted_class'] = [CLASSES[p] for p in pred_cls]
    out['prob_predicted'] = probs[np.arange(len(pred_cls)), pred_cls]
    path = OUT_DIR / f'fold{fold_n}_predictions.csv'
    out.to_csv(path, index=False)
    print(f'  Predictions saved  : {path.name}  ({len(out)} rows)')


def save_fold_training_curves(history: dict, fold_n: int):
    """VERSION 2 CHANGE: save epoch-level training curves for this fold."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Fold {fold_n} Training Curves', fontweight='bold')

    # Combine Stage A + B sequences with a divider
    a_tr  = history.get('A_tr_loss', [])
    b_tr  = history.get('B_tr_loss', [])
    a_kl  = history.get('A_va_kl',   [])
    b_kl  = history.get('B_va_kl',   [])
    a_auc = history.get('A_va_auc',  [])
    b_auc = history.get('B_va_auc',  [])

    split = len(a_tr)
    tr_loss = a_tr + b_tr
    va_kl   = a_kl + b_kl
    va_auc  = a_auc + b_auc

    eps = range(1, len(tr_loss)+1)

    # Train loss
    axes[0].plot(eps, tr_loss, 'steelblue', lw=2)
    if split: axes[0].axvline(split, color='gray', ls=':', alpha=0.7, label='Stage B start')
    axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('KL Loss'); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)

    # Val KL
    if va_kl:
        eps_kl = range(1, len(va_kl)+1)
        axes[1].plot(eps_kl, va_kl, 'crimson', lw=2)
        if split: axes[1].axvline(split, color='gray', ls=':', alpha=0.7)
        axes[1].set_title('Val KL Divergence'); axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('KL'); axes[1].grid(alpha=0.3)

    # Val AUC
    if va_auc:
        eps_auc = range(1, len(va_auc)+1)
        axes[2].plot(eps_auc, va_auc, 'darkgreen', lw=2)
        if split: axes[2].axvline(split, color='gray', ls=':', alpha=0.7)
        axes[2].set_title('Val Macro AUC'); axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('AUC'); axes[2].set_ylim(0, 1); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    path = OUT_DIR / f'training_curves_fold{fold_n}.png'
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.close()
    print(f'  Training curves saved: {path.name}')


# ─────────────────────────────────────────────────────────────────────
# VERSION 2 CHANGE: modified train_full_cv — sampler + diagnostics
# ─────────────────────────────────────────────────────────────────────
def train_full_cv(n_folds: int = N_FOLDS):
    assert eeg_all['patient_id'].isnull().sum() == 0

    groups = eeg_all['patient_id'].values
    labels = eeg_all['expert_consensus'].values
    gkf    = GroupKFold(n_splits=n_folds)
    crit   = KLDivSoftLoss()   # VERSION 2: original KL loss, no weighting

    fold_results      = []
    global_best_kl    = float('inf')
    global_best_state = None

    for fold, (tr_idx, va_idx) in enumerate(
        gkf.split(np.zeros(len(eeg_all)), labels, groups=groups)
    ):
        print(f'\n{"="*50}')
        print(f'  FOLD {fold+1}/{n_folds}')
        print(f'{"="*50}')

        tr_df = eeg_all.iloc[tr_idx].copy()
        va_df = eeg_all.iloc[va_idx].copy()

        overlap = set(tr_df['patient_id']) & set(va_df['patient_id'])
        assert not overlap, f'PATIENT LEAKAGE: {overlap}'

        print(f'  Train: {len(tr_df):,} EEGs | {tr_df["patient_id"].nunique()} patients')
        print(f'  Val  : {len(va_df):,} EEGs | {va_df["patient_id"].nunique()} patients')

        tr_high = tr_df[tr_df['total_votes'] >= MIN_VOTES]
        kw = dict(num_workers=N_WORKERS, pin_memory=DEVICE.type=='cuda')

        # VERSION 2 CHANGE: use WeightedRandomSampler for training loaders
        # Validation uses standard sequential loading (no sampler)
        sampler_hi = make_weighted_sampler(tr_high)
        sampler_al = make_weighted_sampler(tr_df)

        ds_hi = HMSDataset(tr_high, eeg_data, id2idx, augment=True)
        ds_al = HMSDataset(tr_df,   eeg_data, id2idx, augment=True)
        ds_va = HMSDataset(va_df,   eeg_data, id2idx, augment=False)

        ld_hi = DataLoader(ds_hi, BATCH_SIZE, sampler=sampler_hi,
                           drop_last=True, **kw)   # VERSION 2: sampler replaces shuffle
        ld_al = DataLoader(ds_al, BATCH_SIZE, sampler=sampler_al,
                           drop_last=True, **kw)   # VERSION 2: sampler replaces shuffle
        ld_va = DataLoader(ds_va, BATCH_SIZE, shuffle=False, **kw)

        set_seed(42 + fold)
        model   = HMSModel().to(DEVICE)
        scaler  = GradScaler(enabled=USE_AMP)
        history = defaultdict(list)
        best_kl = float('inf')
        best_st = None

        # Stage A: high-consensus data
        A_EPOCHS = MAX_EPOCHS // 2 + MAX_EPOCHS % 2
        print(f'\n  Stage A: {A_EPOCHS} epochs (high-consensus, weighted sampler)')
        opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        sch = OneCycleLR(opt, max_lr=LR, steps_per_epoch=len(ld_hi),
                         epochs=A_EPOCHS, pct_start=0.1)
        es_a = EarlyStopping(PATIENCE, 'min')

        for ep in range(A_EPOCHS):
            tr = train_one_epoch(model, ld_hi, opt, crit, scaler, sch)
            va = eval_one_epoch(model, ld_va, crit)
            history['A_tr_loss'].append(tr['loss'])
            history['A_va_kl'].append(va['kl'])
            history['A_va_auc'].append(va['auc'])  # VERSION 2: track AUC per epoch
            if es_a(va['kl']):
                best_kl = va['kl']; best_st = copy.deepcopy(model.state_dict())
            if (ep+1) % 3 == 0 or ep == 0:
                print(f'    ep{ep+1:02d}  tr={tr["loss"]:.4f}  va_kl={va["kl"]:.4f}  auc={va["auc"]:.4f}')
            if es_a.stop: print(f'    Early stop ep{ep+1}'); break

        model.load_state_dict(best_st)

        # Stage B: all data
        B_EPOCHS = MAX_EPOCHS // 2
        print(f'\n  Stage B: {B_EPOCHS} epochs (all data, weighted sampler, lr/5)')
        opt2 = AdamW(model.parameters(), lr=LR/5, weight_decay=WEIGHT_DECAY)
        sch2 = CosineAnnealingLR(opt2, T_max=B_EPOCHS)
        es_b = EarlyStopping(PATIENCE, 'min')

        for ep in range(B_EPOCHS):
            tr = train_one_epoch(model, ld_al, opt2, crit, scaler, accum=GRAD_ACCUM)
            va = eval_one_epoch(model, ld_va, crit)
            sch2.step()
            history['B_tr_loss'].append(tr['loss'])
            history['B_va_kl'].append(va['kl'])
            history['B_va_auc'].append(va['auc'])  # VERSION 2: track AUC per epoch
            if es_b(va['kl']):
                best_kl = va['kl']; best_st = copy.deepcopy(model.state_dict())
            if (ep+1) % 3 == 0 or ep == 0:
                print(f'    ep{ep+1:02d}  tr={tr["loss"]:.4f}  va_kl={va["kl"]:.4f}  auc={va["auc"]:.4f}')
            if es_b.stop: print(f'    Early stop ep{ep+1}'); break

        model.load_state_dict(best_st)
        final = eval_one_epoch(model, ld_va, crit)

        torch.save({'state': best_st, 'fold': fold+1, 'kl': best_kl, 'auc': final['auc']},
                   MODEL_DIR / f'fold{fold+1}.pt')

        # VERSION 2 CHANGE: save per-fold diagnostics
        save_fold_confusion(final['true_cls'], final['pred_cls'], fold+1)
        save_fold_predictions(va_df.reset_index(drop=True), final['probs'],
                              final['pred_cls'], final['true_cls'], fold+1)
        save_fold_training_curves(dict(history), fold+1)

        fold_results.append({
            'fold':     fold+1,
            'kl':       best_kl,
            'auc':      final['auc'],
            'acc':      final['acc'],
            'probs':    final['probs'],
            'pred_cls': final['pred_cls'],
            'true_cls': final['true_cls'],
            'history':  dict(history),
        })

        print(f'  Fold {fold+1} | KL={best_kl:.4f} | AUC={final["auc"]:.4f}')

        if best_kl < global_best_kl:
            global_best_kl = best_kl; global_best_state = best_st

        del model, ld_hi, ld_al, ld_va, ds_hi, ds_al, ds_va
        gc.collect(); torch.cuda.empty_cache()

    torch.save({'state': global_best_state, 'kl': global_best_kl},
               MODEL_DIR / 'best_global.pt')

    print('\nCROSS-VALIDATION SUMMARY')
    for r in fold_results:
        print(f'  Fold {r["fold"]} | KL={r["kl"]:.4f}  AUC={r["auc"]:.4f}  Acc={r["acc"]:.3f}')
    kls  = [r['kl']  for r in fold_results]
    aucs = [r['auc'] for r in fold_results]
    print(f'  Mean KL  : {np.mean(kls):.4f} +/- {np.std(kls):.4f}')
    print(f'  Mean AUC : {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}')
    return fold_results, global_best_state

print('Training function defined (V2: WeightedRandomSampler + per-fold diagnostics)')


---
## Cell 9 — Evaluation Dashboard
**V2 Changes:**
- `compute_focal_metrics()`: Sensitivity, Specificity, Precision, Recall, F1 per class → `class_metrics.csv` + `class_metrics.png`
- `plot_roc_curves()`: per-class ROC curves → `roc_curves.png`
- Original confusion matrix and dashboard kept


In [ ]:
# ─────────────────────────────────────────────────────────────────
# VERSION 2 CHANGE: focal evaluation metrics helper
# ─────────────────────────────────────────────────────────────────

def compute_focal_metrics(all_true: np.ndarray, all_pred: np.ndarray,
                           all_probs: np.ndarray) -> pd.DataFrame:
    """
    VERSION 2 CHANGE: compute per-class Sensitivity, Specificity,
    Precision, Recall, F1. Saved to class_metrics.csv and class_metrics.png.
    """
    rows = []
    for ci, cls in enumerate(CLASSES):
        tp = ((all_true == ci) & (all_pred == ci)).sum()
        tn = ((all_true != ci) & (all_pred != ci)).sum()
        fp = ((all_true != ci) & (all_pred == ci)).sum()
        fn = ((all_true == ci) & (all_pred != ci)).sum()

        sensitivity = tp / (tp + fn + 1e-9)    # recall
        specificity = tn / (tn + fp + 1e-9)
        precision   = tp / (tp + fp + 1e-9)
        recall      = sensitivity
        f1          = 2 * precision * recall / (precision + recall + 1e-9)

        # AUC
        binary = (all_true == ci).astype(int)
        auc_val = roc_auc_score(binary, all_probs[:, ci]) if 0 < binary.sum() < len(binary) else float('nan')

        rows.append({
            'Class':       cls.upper(),
            'Sensitivity': round(sensitivity, 4),
            'Specificity': round(specificity, 4),
            'Precision':   round(precision,   4),
            'Recall':      round(recall,       4),
            'F1':          round(f1,           4),
            'AUC':         round(auc_val,      4),
            'Support':     int((all_true == ci).sum()),
        })

    df = pd.DataFrame(rows).set_index('Class')
    df.to_csv(OUT_DIR / 'class_metrics.csv')
    print('\nFocal Metrics (saved to class_metrics.csv):')
    print(df.to_string())

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Per-Class Focal Metrics (V2)', fontweight='bold', fontsize=13)

    metric_cols = ['Sensitivity', 'Specificity', 'Precision', 'F1']
    x = np.arange(len(CLASSES))
    w = 0.18
    colors_m = ['#E74C3C', '#27AE60', '#2980B9', '#F39C12']
    for mi, (met, col) in enumerate(zip(metric_cols, colors_m)):
        axes[0].bar(x + mi*w - 1.5*w, df[met].values, w, label=met, color=col, alpha=0.85)
    axes[0].set_xticks(x); axes[0].set_xticklabels(df.index, fontsize=9)
    axes[0].set_ylim(0, 1.15); axes[0].set_title('Sensitivity / Specificity / Precision / F1')
    axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

    # AUC bar
    auc_vals = df['AUC'].values
    bar_colors = ['#E74C3C' if c in ('SEIZURE','LRDA') else 'steelblue' for c in df.index]
    axes[1].bar(df.index, auc_vals, color=bar_colors, alpha=0.85, edgecolor='white')
    axes[1].set_ylim(0, 1.05); axes[1].set_title('Per-Class AUC (red = minority target classes)')
    axes[1].set_ylabel('AUC'); axes[1].grid(axis='y', alpha=0.3)
    for i, v in enumerate(auc_vals):
        if not np.isnan(v):
            axes[1].text(i, v+0.01, f'{v:.3f}', ha='center', fontsize=8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'class_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('class_metrics.png saved')
    return df


# ─────────────────────────────────────────────────────────────────
# VERSION 2 CHANGE: per-class ROC curves
# ─────────────────────────────────────────────────────────────────

def plot_roc_curves(all_true: np.ndarray, all_probs: np.ndarray):
    """
    VERSION 2 CHANGE: generate one ROC curve per class, saved to roc_curves.png.
    """
    colors = ['#E74C3C','#E67E22','#F39C12','#27AE60','#2980B9','#8E44AD']
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle('Per-Class ROC Curves (V2)', fontweight='bold', fontsize=13)

    for ci, (cls, col) in enumerate(zip(CLASSES, colors)):
        ax = axes[ci // 3][ci % 3]
        binary = (all_true == ci).astype(int)
        if 0 < binary.sum() < len(binary):
            fpr, tpr, _ = roc_curve(binary, all_probs[:, ci])
            auc_val      = sklearn_auc(fpr, tpr)
            ax.plot(fpr, tpr, color=col, lw=2.5,
                    label=f'AUC = {auc_val:.3f}')
            ax.fill_between(fpr, tpr, alpha=0.12, color=col)
        ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
        ax.set_title(f'{cls.upper()}', fontweight='bold',
                     color='#C0392B' if cls in ('seizure','lrda') else 'black')
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
        ax.set_xlim(0,1); ax.set_ylim(0,1.02)
        ax.legend(fontsize=9); ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('roc_curves.png saved')


# ─────────────────────────────────────────────────────────────────
# evaluate_all_folds: original + V2 additions called at the end
# ─────────────────────────────────────────────────────────────────

def evaluate_all_folds(fold_results):
    all_pred  = np.concatenate([r['pred_cls'] for r in fold_results])
    all_true  = np.concatenate([r['true_cls'] for r in fold_results])
    all_probs = np.vstack([r['probs'] for r in fold_results])

    cm = confusion_matrix(all_true, all_pred)
    cm_norm = cm.astype(float) / cm.sum(1, keepdims=True).clip(min=1)

    rpt = classification_report(all_true, all_pred,
                                 target_names=[c.upper() for c in CLASSES],
                                 output_dict=True, zero_division=0)
    metrics_df = pd.DataFrame({
        c.upper(): {
            'Sensitivity': rpt[c.upper()]['recall'],
            'Precision':   rpt[c.upper()]['precision'],
            'F1':          rpt[c.upper()]['f1-score'],
        } for c in CLASSES
    }).T

    auc_dict = {}
    for ci, c in enumerate(CLASSES):
        b = (all_true == ci).astype(int)
        if 0 < b.sum() < len(b):
            auc_dict[c.upper()] = roc_auc_score(b, all_probs[:, ci])
    metrics_df['AUC'] = pd.Series(auc_dict)

    print('Clinical Metrics (aggregated over all folds):')
    print(metrics_df.round(3).to_string())

    # Original dashboard figure
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle('HMS EEG Model — Evaluation Dashboard (V2)', fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.4)
    cls_U  = [c.upper() for c in CLASSES]
    colors = ['#E74C3C','#E67E22','#F39C12','#27AE60','#2980B9','#8E44AD']

    ax = fig.add_subplot(gs[0, :2])
    im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(cls_U, rotation=15)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(cls_U)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Normalised Confusion Matrix')
    plt.colorbar(im, ax=ax, fraction=0.04)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            v = cm_norm[i,j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=10,
                    color='white' if v > 0.5 else 'black')

    ax2 = fig.add_subplot(gs[0, 2])
    xi  = np.arange(N_CLASSES); w = 0.25
    ax2.bar(xi-w, metrics_df['Sensitivity'].values, w, label='Sensitivity', color='steelblue', alpha=0.85)
    ax2.bar(xi,   metrics_df['F1'].values, w, label='F1', color='crimson', alpha=0.85)
    ax2.bar(xi+w, metrics_df['AUC'].fillna(0).values, w, label='AUC', color='#27AE60', alpha=0.85)
    ax2.set_xticks(xi); ax2.set_xticklabels(cls_U, fontsize=8)
    ax2.set_ylim(0, 1.05); ax2.set_title('Per-Class Metrics')
    ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

    best = min(fold_results, key=lambda r: r['kl'])
    ax3  = fig.add_subplot(gs[1, 0])
    h    = best['history']
    if 'A_va_kl' in h:
        ax3.plot(h['A_va_kl'], 'b-', lw=2, label='Stage A val KL')
    if 'B_va_kl' in h:
        off = len(h.get('A_va_kl', []))
        ax3.plot(range(off, off+len(h['B_va_kl'])), h['B_va_kl'], 'r-', lw=2, label='Stage B val KL')
        ax3.axvline(off, color='gray', ls=':', alpha=0.5)
    ax3.set_title(f'Best Fold {best["fold"]} KL Curve')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('KL Divergence')
    ax3.legend(); ax3.grid(alpha=0.3)

    ax4  = fig.add_subplot(gs[1, 1])
    fns  = [r['fold'] for r in fold_results]
    kls  = [r['kl']   for r in fold_results]
    aus  = [r['auc']  for r in fold_results]
    ax4.bar(fns, kls, color='steelblue', alpha=0.7)
    ax4b = ax4.twinx()
    ax4b.plot(fns, aus, 'ro-', lw=2, ms=6)
    ax4.set_xlabel('Fold'); ax4.set_ylabel('KL Div', color='steelblue')
    ax4b.set_ylabel('AUC', color='red')
    ax4.set_title('Per-Fold Performance')

    ax5 = fig.add_subplot(gs[1, 2])
    for ci, (c, col) in enumerate(zip(CLASSES, colors)):
        m = all_true == ci
        if m.sum() > 0:
            ax5.hist(all_probs[m, ci], bins=20, alpha=0.5, color=col, label=c.upper(), density=True)
    ax5.set_xlabel('Predicted P (true class)')
    ax5.set_ylabel('Density'); ax5.set_title('Calibration')
    ax5.legend(fontsize=7)

    plt.savefig(OUT_DIR/'evaluation.png', dpi=150, bbox_inches='tight')
    plt.show()
    metrics_df.to_csv(OUT_DIR/'metrics.csv')
    print('Original evaluation.png saved')

    # VERSION 2 CHANGE: call focal metrics + ROC curves
    focal_df = compute_focal_metrics(all_true, all_pred, all_probs)
    plot_roc_curves(all_true, all_probs)

    return metrics_df, focal_df

print('Evaluation functions defined (V2: focal metrics + ROC curves)')


---
## Cell 10 — Patient Adapter Fine-Tuning
Backbone frozen. ~1.5% of parameters trained per patient. **Unchanged from V1.**

In [ ]:
class Adapter(nn.Module):
    def __init__(self, d: int = 256, bottleneck: int = ADAPTER_DIM):
        super().__init__()
        self.down  = nn.Linear(d, bottleneck)
        self.act   = nn.GELU()
        self.up    = nn.Linear(bottleneck, d)
        self.scale = nn.Parameter(torch.zeros(1))
        nn.init.kaiming_normal_(self.down.weight)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x): return x + self.scale * self.up(self.act(self.down(x)))


class AdaptedModel(nn.Module):
    def __init__(self, base: HMSModel):
        super().__init__()
        lstm_out = LSTM_HIDDEN * 2
        self.cnn  = base.cnn
        self.lstm = base.lstm
        self.attn = base.attn
        self.head = base.head

        for mod in [self.cnn, self.lstm, self.attn, self.head]:
            for p in mod.parameters(): p.requires_grad_(False)

        self.adp_lstm = Adapter(lstm_out, ADAPTER_DIM)
        self.adp_attn = Adapter(lstm_out, ADAPTER_DIM)
        self.pat_gate = nn.Sequential(
            nn.LayerNorm(lstm_out),
            nn.Linear(lstm_out, 32), nn.GELU(), nn.Linear(32, N_CLASSES)
        )
        tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        tt = sum(p.numel() for p in self.parameters())
        print(f'  Adapter trainable: {tr:,} / {tt:,}  ({100*tr/tt:.1f}%)')

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        x, _ = self.lstm(x)
        x = self.adp_lstm(x)
        x, _ = self.attn(x)
        x = self.adp_attn(x)
        x = x.mean(1)
        return self.head(x) + 0.3 * self.pat_gate(x)


def adapt_all_patients(meta_df: pd.DataFrame, model_path: Path):
    ckpt = torch.load(model_path, map_location='cpu')
    base = HMSModel(); base.load_state_dict(ckpt['state'])
    crit = KLDivSoftLoss(); results = {}

    for pid in tqdm(meta_df['patient_id'].unique(), desc='Adapting patients'):
        pat = meta_df[meta_df['patient_id'] == pid]
        if len(pat) < 4: continue
        cut = max(1, int(0.8*len(pat)))
        tr_df, va_df = pat.iloc[:cut], pat.iloc[cut:]

        ds_tr = HMSDataset(tr_df, eeg_data, id2idx, augment=True)
        ds_va = HMSDataset(va_df, eeg_data, id2idx, augment=False)
        if len(ds_tr) == 0 or len(ds_va) == 0: continue

        ld_tr = DataLoader(ds_tr, min(8, len(ds_tr)), shuffle=True,  num_workers=0)
        ld_va = DataLoader(ds_va, min(8, len(ds_va)), shuffle=False, num_workers=0)

        pre = eval_one_epoch(base.to(DEVICE), ld_va, crit); base.cpu()

        set_seed(42)
        adp    = AdaptedModel(copy.deepcopy(base)).to(DEVICE)
        opt    = AdamW([p for p in adp.parameters() if p.requires_grad],
                       lr=ADAPTER_LR, weight_decay=WEIGHT_DECAY)
        scaler = GradScaler(enabled=USE_AMP)
        best_kl, best_st = float('inf'), None

        for _ in range(ADAPTER_EPOCHS):
            adp.train()
            for x, y, _ in ld_tr:
                x, y = x.to(DEVICE), y.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                with autocast(enabled=USE_AMP):
                    loss = crit(adp(x), y)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in adp.parameters() if p.requires_grad], 1.0)
                scaler.step(opt); scaler.update()
            post = eval_one_epoch(adp, ld_va, crit)
            if post['kl'] < best_kl:
                best_kl = post['kl']; best_st = copy.deepcopy(adp.state_dict())

        results[pid] = {
            'kl_global': pre['kl'], 'kl_adapted': best_kl,
            'delta': pre['kl'] - best_kl, 'n': len(ds_tr)
        }
        del adp; gc.collect(); torch.cuda.empty_cache()

    df_r = pd.DataFrame(results).T.astype(float)
    improved = (df_r['delta'] > 0).sum()
    print(f'Patients adapted : {len(df_r)}')
    print(f'Improved         : {improved}  ({100*improved/max(1,len(df_r)):.0f}%)')
    print(f'Mean KL global   : {df_r["kl_global"].mean():.4f}')
    print(f'Mean KL adapted  : {df_r["kl_adapted"].mean():.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Patient Adaptation Results', fontweight='bold')
    axes[0].hist(df_r['delta'], bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(0, color='red', lw=2, label='No improvement')
    axes[0].axvline(df_r['delta'].mean(), color='green', lw=2, label=f'Mean {df_r["delta"].mean():.3f}')
    axes[0].set_xlabel('KL Improvement'); axes[0].legend()
    lim = max(df_r['kl_global'].max(), df_r['kl_adapted'].max())
    axes[1].scatter(df_r['kl_global'], df_r['kl_adapted'], c=df_r['delta'], cmap='RdYlGn', alpha=0.7)
    axes[1].plot([0,lim],[0,lim],'k--', alpha=0.4, label='No change')
    axes[1].set_xlabel('KL global'); axes[1].set_ylabel('KL adapted')
    axes[1].set_title('Below diagonal = improved'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR/'adaptation.png', dpi=150, bbox_inches='tight')
    plt.show()
    df_r.to_csv(OUT_DIR/'adaptation_metrics.csv')
    return df_r

print('Adapter functions defined')

---
## Cell 11 — Attention Explainability
Attention weights are (B, heads, T, T). Average both axes to get (T,) temporal saliency. **Unchanged from V1.**

In [ ]:
SEVERITY = {
    'seizure': 'CRITICAL', 'lpd': 'HIGH', 'gpd': 'HIGH',
    'lrda': 'MODERATE', 'grda': 'MODERATE', 'other': 'LOW'
}

def explain_eeg(model: HMSModel, eeg_arr: np.ndarray,
                true_label: str = None, eeg_id: int = None) -> dict:
    model.eval()
    x = torch.from_numpy(eeg_arr[None]).to(DEVICE)
    with torch.no_grad():
        logits, w = model(x, return_attn=True)

    probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    # ATTENTION DIM FIX: w is (1, heads, T, T)
    attn_map = w[0].cpu().float().numpy()           # (heads, T, T)
    attn_seq = attn_map.mean(axis=0).mean(axis=0)   # (T,)

    pred_idx  = probs.argmax()
    pred_cls  = CLASSES[pred_idx]
    pred_prob = probs[pred_idx]
    entropy   = float(-np.sum(probs * np.log(probs + 1e-8)))
    confidence = 1.0 - entropy / np.log(N_CLASSES)

    peak_t   = attn_seq.argmax()
    downsamp = LABEL_SAMP // len(attn_seq)
    peak_sec = float(peak_t * downsamp) / SFREQ

    ps = max(0, peak_t*downsamp-50); pe = min(LABEL_SAMP, peak_t*downsamp+50)
    region_act = {
        'LL': float(np.abs(eeg_arr[0:4,  ps:pe]).mean()),
        'LP': float(np.abs(eeg_arr[4:8,  ps:pe]).mean()),
        'RP': float(np.abs(eeg_arr[8:12, ps:pe]).mean()),
        'RL': float(np.abs(eeg_arr[12:,  ps:pe]).mean()),
    }
    return {
        'pred_cls': pred_cls, 'pred_prob': pred_prob,
        'probs': dict(zip(CLASSES, probs.tolist())),
        'confidence': confidence, 'severity': SEVERITY[pred_cls],
        'attn': attn_seq, 'peak_sec': peak_sec,
        'top_region': max(region_act, key=region_act.get),
        'region_act': region_act, 'true_label': true_label
    }


def plot_explanation(eeg_arr: np.ndarray, res: dict, eeg_id=None):
    fig = plt.figure(figsize=(20, 11), facecolor='#0D1117')
    fig.suptitle(
        f'EEG {eeg_id}  |  Pred: {res["pred_cls"].upper()} '
        f'({res["pred_prob"]*100:.1f}%)  |  Severity: {res["severity"]}',
        color='white', fontsize=12, fontweight='bold', y=0.98
    )
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35,
                           left=0.06, right=0.97, top=0.93, bottom=0.07)
    t  = np.linspace(0, LABEL_SEC, LABEL_SAMP)
    ta = np.linspace(0, LABEL_SEC, len(res['attn']))
    rcols  = ['#E74C3C','#27AE60','#2980B9','#8E44AD']
    rnames = ['LL','LP','RP','RL']

    for ri in range(4):
        ax = fig.add_subplot(gs[ri//2, ri%2])
        ax.set_facecolor('#161B22')
        for ci in range(4):
            ax.plot(t, eeg_arr[ri*4+ci]+ci*3.5, lw=0.6, alpha=0.9, color=rcols[ri])
        ax.axvline(res['peak_sec'], color='yellow', lw=1.5, ls='--', alpha=0.8)
        ax.set_title(f'{rnames[ri]} Region', color='white', fontsize=9)
        ax.tick_params(colors='white', labelsize=7); ax.set_yticks([])
        [sp.set_color('gray') for sp in ax.spines.values()]

    ax_a = fig.add_subplot(gs[0:2, 2]); ax_a.set_facecolor('#161B22')
    ax_a.fill_between(ta, res['attn'], alpha=0.6, color='#F39C12')
    ax_a.plot(ta, res['attn'], color='#F39C12', lw=1.5)
    ax_a.axvline(res['peak_sec'], color='white', lw=1.5, ls='--',
                 label=f'Peak @ {res["peak_sec"]:.1f}s')
    ax_a.set_title('Temporal Attention', color='white', fontsize=9)
    ax_a.legend(fontsize=8, labelcolor='white', facecolor='#161B22')
    ax_a.tick_params(colors='white', labelsize=7)
    [sp.set_color('gray') for sp in ax_a.spines.values()]

    ax_p = fig.add_subplot(gs[2, :2]); ax_p.set_facecolor('#161B22')
    ccols = ['#E74C3C','#E67E22','#F39C12','#27AE60','#2980B9','#8E44AD']
    vals  = [res['probs'][c] for c in CLASSES]
    bars  = ax_p.barh([c.upper() for c in CLASSES], vals,
                      color=ccols, alpha=0.85, edgecolor='white', lw=0.3)
    bars[CLASSES.index(res['pred_cls'])].set_linewidth(2.5)
    for bar, v in zip(bars, vals):
        ax_p.text(min(v+0.01, 0.94), bar.get_y()+bar.get_height()/2,
                  f'{v*100:.1f}%', va='center', color='white', fontsize=9)
    ax_p.set_xlim(0, 1.05); ax_p.set_title('Class Probabilities', color='white')
    ax_p.tick_params(colors='white')
    [sp.set_color('gray') for sp in ax_p.spines.values()]

    ax_s = fig.add_subplot(gs[2, 2]); ax_s.set_facecolor('#161B22'); ax_s.axis('off')
    info = [
        ('PREDICTION', res['pred_cls'].upper(), '#F39C12'),
        ('PROBABILITY', f'{res["pred_prob"]*100:.1f}%', '#27AE60'),
        ('CONFIDENCE', f'{res["confidence"]*100:.0f}%', '#27AE60'),
        ('SEVERITY', res['severity'], 'white'),
        ('PEAK FOCUS', f'{res["peak_sec"]:.1f}s', '#2980B9'),
        ('TOP REGION', res['top_region'], '#8E44AD'),
        ('TRUE LABEL', (res['true_label'] or 'N/A').upper(), '#AAAAAA'),
    ]
    for i, (lbl, val, col) in enumerate(info):
        y = 0.93 - i*0.135
        ax_s.text(0.05, y,      lbl, color='#666', fontsize=8, fontweight='bold', transform=ax_s.transAxes)
        ax_s.text(0.05, y-0.05, val, color=col,   fontsize=11, fontweight='bold', transform=ax_s.transAxes)

    plt.savefig(OUT_DIR/f'explain_{eeg_id}.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
    plt.show()

print('Explainability functions defined')

---
## Cell 12 — One-Click Full Pipeline Runner
Runs all stages. V2 outputs include focal metrics, per-fold confusion matrices, prediction CSVs, training curves, and ROC curves.

In [ ]:
print('='*60)
print(' HMS PATIENT-ADAPTIVE EEG CLASSIFICATION — VERSION 2')
print(' CNN-BiLSTM-Attention + WeightedSampler + Minority Augmentation')
print('='*60)

if not smoke_ok:
    raise RuntimeError('Smoke test failed — re-run Cell 7')

# 1: Train (V2: sampler + per-fold diagnostics)
print('\n[1/4] GroupKFold training (V2)...')
fold_results, best_state = train_full_cv(n_folds=N_FOLDS)

# 2: Evaluate (V2: focal metrics + ROC)
print('\n[2/4] Evaluation (V2)...')
metrics_df, focal_df = evaluate_all_folds(fold_results)

# 3: Patient adapters
print('\n[3/4] Patient adapter fine-tuning...')
adapt_df = adapt_all_patients(eeg_high, MODEL_DIR/'best_global.pt')

# 4: Explainability
print('\n[4/4] Explainability examples...')
ckpt  = torch.load(MODEL_DIR/'best_global.pt', map_location=DEVICE)
model = HMSModel().to(DEVICE)
model.load_state_dict(ckpt['state'])

for cls_name in ['seizure', 'lrda', 'gpd', 'other']:
    rows = eeg_meta[eeg_meta['expert_consensus'] == cls_name]
    if len(rows) == 0: continue
    row = rows.iloc[0]
    arr = prep.load_and_process(int(row['eeg_id']))
    if arr is None: continue
    res = explain_eeg(model, arr, true_label=cls_name, eeg_id=int(row['eeg_id']))
    plot_explanation(arr, res, eeg_id=int(row['eeg_id']))
    print(f'  {cls_name.upper():8s}: pred={res["pred_cls"].upper()} ({res["pred_prob"]*100:.1f}%)')

print('\n' + '='*60)
print(' PIPELINE COMPLETE — V2 OUTPUT FILES')
print('='*60)

# VERSION 2 CHANGE: extended output listing
v2_outputs = [
    'evaluation.png',          'metrics.csv',
    'class_metrics.csv',       'class_metrics.png',    # V2
    'roc_curves.png',                                   # V2
    'fold1_confusion.png',     'fold2_confusion.png',  # V2
    'fold3_confusion.png',                              # V2
    'fold1_predictions.csv',   'fold2_predictions.csv', # V2
    'fold3_predictions.csv',                            # V2
    'training_curves_fold1.png', 'training_curves_fold2.png', # V2
    'training_curves_fold3.png',                        # V2
    'adaptation_metrics.csv',  'adaptation.png',
    'audit.png',               'sample_eeg.png',
]

print(f'\nOutputs in {OUT_DIR}:')
for fname in v2_outputs:
    p = OUT_DIR / fname
    tag = '[V2]' if any(k in fname for k in ['class_metrics','roc_curves','confusion','predictions','training_curves']) else '    '
    size = f'{p.stat().st_size/1e3:6.0f} KB' if p.exists() else '  (not generated)' 
    print(f'  {tag} {fname:<38s}  {size}')


In [ ]:
import os
import shutil
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/HMS_V2_EXPORT")
SAVE_DIR.mkdir(exist_ok=True)

# Model checkpoints
files_to_save = [
    "/kaggle/working/models/best_global.pt",
    "/kaggle/working/models/fold1.pt",
    "/kaggle/working/models/fold2.pt",
    "/kaggle/working/models/fold3.pt",
]

# Evaluation artifacts
extra_files = [
    "/kaggle/working/evaluation.png",
    "/kaggle/working/adaptation.png",
    "/kaggle/working/fold1_confusion.png",
    "/kaggle/working/fold2_confusion.png",
    "/kaggle/working/fold3_confusion.png",
]
extra_files += [
    "/kaggle/working/metrics.csv",
    "/kaggle/working/adaptation_metrics.csv",
    "/kaggle/working/fold1_predictions.csv",
    "/kaggle/working/fold2_predictions.csv",
    "/kaggle/working/fold3_predictions.csv",
    "/kaggle/working/training_curves_fold1.png",
    "/kaggle/working/training_curves_fold2.png",
    "/kaggle/working/training_curves_fold3.png",
]
# Copy everything that exists
for f in files_to_save + extra_files:
    if os.path.exists(f):
        shutil.copy2(f, SAVE_DIR)
        print(f"✓ Copied: {Path(f).name}")
    else:
        print(f"✗ Not found: {f}")

# Create zip
shutil.make_archive(
    "/kaggle/working/HMS_V2_EXPORT",
    "zip",
    "/kaggle/working/HMS_V2_EXPORT"
)

print("\nZIP created:")
print("/kaggle/working/HMS_V2_EXPORT.zip")